# Main empirical study: Lalonde NSW treated ATT and ATE sensitivity

The primary estimand is the standard ATT for NSW treated units.  ATE is also reported as a sensitivity analysis because the GRR wrapper supports it.  Benchmark error is computed for ATT when the randomized NSW experimental benchmark is available.

In [ ]:
from pathlib import Path
import sys
import os
import warnings
warnings.filterwarnings("once")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Make the experiment helper importable whether Jupyter is launched from the
# repository root or from notebooks/experiments.
for _candidate in [Path.cwd(), Path.cwd() / "notebooks" / "experiments"]:
    if (_candidate / "grr_experiment_utils.py").exists() and str(_candidate) not in sys.path:
        sys.path.insert(0, str(_candidate))

# Local experiment helpers. These live beside the notebooks and do not modify src/genriesz.
from grr_experiment_utils import *
from genriesz import (
    grr_ate, grr_att, ATEFunctional, ATTFunctional,
    BregmanGenerator, SquaredGenerator, UKLGenerator, BKLGenerator, BPGenerator,
    PolynomialBasis, TreatmentInteractionBasis,
)

# Optional random-forest leaf basis used in model-comparison experiments.
try:
    from sklearn.ensemble import RandomForestRegressor
    from genriesz.sklearn_basis import RandomForestLeafBasis
    SKLEARN_AVAILABLE = True
except Exception as exc:
    SKLEARN_AVAILABLE = False
    print("scikit-learn is not available; random-forest basis cells will fall back to RKHS.", exc)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

In [ ]:
FAST_MODE = True
POLYNOMIAL_DEGREE = 1 if FAST_MODE else 2
DOWNLOAD_DATA = False if FAST_MODE else True
os.environ["GRR_ALLOW_REMOTE_DATA"] = "1" if DOWNLOAD_DATA else "0"

# Every analysis estimates both targets when the wrapper supports them.
# If one target fails for a particular method, the failure row is kept and the other target is displayed.
ESTIMANDS = ("ate", "att")

N_REPS = 1 if FAST_MODE else 200
N = 40 if FAST_MODE else 3000
FOLDS = 2 if FAST_MODE else 5
MAX_ITER = 25 if FAST_MODE else 500
N_FEATURES = 4 if FAST_MODE else 120

LOSS_GRID = [("SQ", None), ("UKL", None), ("BKL", None), ("BP", 0.5)]
LAMBDA_MAIN = 1e-2
ESTIMATORS_ALL = ("ra", "rw", "arw", "tmle")
print(display_mode_banner(FAST_MODE))
CONTROL_SOURCES = ["cps3"] if FAST_MODE else ["cps3", "psid3"]
TABLE_TITLE_LALONDE = "Table E-main-2. Lalonde NSW estimates for ATE and ATT."
FIGURE_TITLE_LALONDE = "Lalonde estimates by estimand and loss"
FIGURE_TITLE_LALONDE_BENCHMARK = "Lalonde ATT benchmark error"

In [ ]:
def dgp_factory(name, *, n, seed):
    """Three DGPs used throughout the main simulation study."""
    if name == "DGP1 nonlinear heterogeneous":
        return make_ate_data(n=n, d=8, kappa=1.0, heterogeneous=True, seed=seed, noise_sd=1.0)
    if name == "DGP2 weak overlap":
        return make_ate_data(n=n, d=8, kappa=2.5, heterogeneous=True, seed=seed, noise_sd=1.0)
    if name == "DGP3 Kang-Schafer misspecification":
        return make_kang_schafer_data(n=n, seed=seed, tau=1.0)
    raise ValueError(name)


def basis_for_model(model, *, seed=0, n_features=N_FEATURES, sigma=1.0):
    """Editable basis map used by the GRR experiments."""
    model = str(model).lower()
    if model in {"rkhs", "gaussian"}:
        return make_treatment_basis("rkhs", n_features=n_features, sigma=sigma, seed=seed)
    if model in {"poly", "polynomial"}:
        return make_treatment_basis("poly", degree=POLYNOMIAL_DEGREE, n_features=n_features, sigma=sigma, seed=seed)
    if model in {"rff", "fourier", "random_fourier"}:
        return make_treatment_basis("rff", n_features=n_features, sigma=sigma, seed=seed)
    if model in {"rf", "random_forest"}:
        if SKLEARN_AVAILABLE:
            rf = RandomForestRegressor(n_estimators=8 if FAST_MODE else 80, max_depth=3 if FAST_MODE else 5,
                                       min_samples_leaf=5, random_state=seed)
            return TreatmentInteractionBasis(base_basis=RandomForestLeafBasis(rf, include_bias=True, normalize=True))
        return make_treatment_basis("rkhs", n_features=n_features, sigma=sigma, seed=seed)
    raise ValueError(model)


def loss_label(loss, omega=None):
    return loss if omega is None else f"{loss}({omega:g})"


def add_metadata(df, **kwargs):
    out = df.copy()
    for k, v in kwargs.items():
        out[k] = v
    return out


def clean_results(df):
    if "status" in df.columns:
        status = df["status"].fillna("ok")
    else:
        status = pd.Series("ok", index=df.index)
    return df[status.eq("ok") & df["estimator"].ne("failed")].copy()


def mc_table(df, group_cols, estimator_filter=None):
    d = clean_results(df)
    if estimator_filter is not None:
        d = d[d["estimator"].isin(list(estimator_filter))]
    return summarize_mc(d, group_cols)


def boxplot_metric(df, *, group_col, metric, title, estimator="arw", rotate=45, ylim=None):
    d = clean_results(df)
    if estimator is not None:
        d = d[d["estimator"] == estimator]
    labels = list(d[group_col].dropna().astype(str).unique())
    data = [d.loc[d[group_col].astype(str) == lab, metric].dropna().to_numpy() for lab in labels]
    fig, ax = plt.subplots(figsize=(max(8, 0.5 * len(labels)), 4.5))
    ax.boxplot(data, labels=labels, showmeans=True)
    ax.set_title(title)
    ax.set_xlabel(group_col)
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=rotate)
    if ylim is not None:
        ax.set_ylim(*ylim)
    fig.tight_layout()
    plt.show()
    return fig, ax


def run_one(data, *, estimand, loss, omega, basis, lam, cross_fit, folds, estimators=ESTIMATORS_ALL,
            penalty="l2", max_iter=MAX_ITER):
    """Fit ATE or ATT. Failures are returned as rows so tables remain complete."""
    try:
        df = fit_grr_estimand(
            data, estimand=estimand, loss=loss, omega=omega, basis=basis, lam=lam,
            cross_fit=cross_fit, folds=folds, estimators=estimators, penalty=penalty,
            max_iter=max_iter,
        )
        df["status"] = "ok"
        return df
    except Exception as exc:
        theta = true_theta_for_estimand(data, estimand)
        row = {
            "estimand": estimand.upper(), "estimator": "failed", "status": type(exc).__name__,
            "message": str(exc)[:240], "true_theta": theta,
        }
        return pd.DataFrame([row])

In [ ]:
rows = []
for control_source in CONTROL_SOURCES:
    data = load_lalonde(control_source=control_source, fallback_seed=6000)
    data = subsample_data(data, n=500 if FAST_MODE else len(data["X"]), seed=1)
    df = fit_ate_att_dataset(
        data,
        estimands=ESTIMANDS,
        loss_grid=LOSS_GRID,
        lambdas=[LAMBDA_MAIN],
        basis_kind="rkhs",
        cross_fit=True,
        folds=FOLDS,
        estimators=ESTIMATORS_ALL,
        max_iter=MAX_ITER,
    )
    df["control_source"] = control_source
    df["dataset_name"] = data.get("name", "Lalonde")
    rows.append(df)

lalonde_results = pd.concat(rows, ignore_index=True)
print(TABLE_TITLE_LALONDE)
cols = ["estimand", "control_source", "source", "loss", "estimator", "estimate", "se", "ci_low", "ci_high", "benchmark", "benchmark_error", "alpha_abs_p95", "max_abs_smd_weighted", "ess_treated", "ess_control"]
cols = [c for c in cols if c in lalonde_results.columns]
display(safe_display_frame(clean_results(lalonde_results)[cols], n=120))

In [ ]:
plot_df = clean_results(lalonde_results).copy()
plot_df = plot_df[plot_df["estimator"] == "arw"].copy()
plot_df["method"] = plot_df["estimand"] + " | " + plot_df["loss"]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(plot_df["method"], plot_df["estimate"], s=60)
ax.set_title(FIGURE_TITLE_LALONDE)
ax.set_xlabel("Estimand | loss")
ax.set_ylabel("Estimate")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

bench_df = plot_df[(plot_df["estimand"] == "ATT") & ("benchmark_error" in plot_df.columns)]
if "benchmark_error" in plot_df.columns and len(bench_df):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(bench_df["loss"], bench_df["benchmark_error"])
    ax.axhline(0.0, linewidth=1)
    ax.set_title(FIGURE_TITLE_LALONDE_BENCHMARK)
    ax.set_xlabel("Loss")
    ax.set_ylabel("ATT estimate minus experimental benchmark")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    plt.show()

In [ ]:
# Love plot for one representative ATT fit. Edit the method selection here if needed.
try:
    data = load_lalonde(control_source=CONTROL_SOURCES[0], fallback_seed=6000)
    data = subsample_data(data, n=500 if FAST_MODE else len(data["X"]), seed=1)
    basis = basis_for_model("rkhs", seed=0, n_features=N_FEATURES)
    gen = make_generator("SQ")
    res = grr_att(X=data["X"], Y=data["Y"], basis=basis, generator=gen, cross_fit=True, folds=FOLDS,
                  riesz_lam=LAMBDA_MAIN, estimators=("rw", "arw"), max_iter=MAX_ITER)
    love = res.love_plot_data(as_pandas=True)
    display(safe_display_frame(love, n=20))
    fig, ax = res.love_plot(threshold=0.1, max_covariates=20)
    ax.set_title("Lalonde ATT Love plot: SQ-Riesz")
    fig.tight_layout()
    plt.show()
except Exception as exc:
    print("Love plot skipped:", type(exc).__name__, exc)